# Exp 6: Baseline + Image Tiling (Anti-Overfitting)

**Mục tiêu**: Áp dụng kỹ thuật Image Tiling lên baseline (YOLOv8n, 50 epoch),
với các biện pháp chống overfitting được tích hợp trực tiếp vào pipeline tiling.

## Những thay đổi so với baseline

| | Baseline | Exp 6 (Tiling) |
|---|---|---|
| Dataset | `processed/` (ảnh gốc) | `processed_tiled/` (tiles 640×640) |
| Model | YOLOv8n | YOLOv8n |
| Epochs | 50 | 50 (giữ nguyên) |
| imgsz | 640 | 640 (giữ nguyên) |
| batch | 16 | 16 (giữ nguyên) |

## Anti-Overfitting áp dụng trong `tiling.py`

| Biện pháp | Config | Lý do |
|---|---|---|
| `INCLUDE_EMPTY = False` | Bỏ hoàn toàn tiles trống | Loại bỏ background noise, giảm ~85% tiles vô ích |
| `MAX_EMPTY_RATIO = 0.5` | Dự phòng nếu bật empty | Tối đa 1 tile trống / 2 tiles có object |
| Val/Test không bị cap | Chỉ áp dụng cho train | Đảm bảo đánh giá công bằng, không lọc test data |
| `EMPTY_SAMPLE_SEED = 42` | Reproducible sampling | Kết quả nhất quán qua các lần chạy |

## Bước 0: Setup môi trường (Kaggle)

In [ ]:
!git clone https://github.com/Shiba-dotcom/waste-detection_project.git

In [ ]:
!pip install gdown
!gdown --id "17Nmi2fonq31N1PZUlCZ0q7FsXf2JzGqM" -O raw.zip
!mkdir -p /kaggle/working/waste-detection_project/data
!unzip -q raw.zip -d /kaggle/working/waste-detection_project/data/
!rm raw.zip
!rm -rf /kaggle/working/waste-detection_project/.git

In [ ]:
%cd /kaggle/working/waste-detection_project
!pip install -r requirements.txt

## Bước 1–3: Data Cleaning → YOLO Conversion → Split

In [ ]:
%cd /kaggle/working/waste-detection_project/src/data_prep
!python data_cleaning.py

%cd /kaggle/working/waste-detection_project/src
!python Training_dataYolo.py

%cd /kaggle/working/waste-detection_project/src/data_prep
!python split_dataset.py

## Bước 4: Tiling với Anti-Overfitting

File `tiling.py` đã được cập nhật với các biện pháp chống overfitting.

### Vấn đề của config cũ (`INCLUDE_EMPTY = True`)

```
Ảnh TACO ~3024×4032 px → ~48 tiles/ảnh
1048 ảnh train × 48 tiles = ~50,000 tiles
Trong đó: ~85–95% tiles là background THUẦN (không có object nào)
→ Mô hình 50 epoch 'thấy' dữ liệu như thể chạy ~2400 epoch thực tế!
```

### Config mới (`INCLUDE_EMPTY = False`)

```
Chỉ giữ tiles CÓ OBJECT → giảm ~85% số tiles
Hệ số mở rộng thực tế: ~5–10x thay vì ~48x
Val/Test không bị ảnh hưởng (đánh giá trên toàn bộ tiles)
```

In [ ]:
%cd /kaggle/working/waste-detection_project/src/data_prep
!python tiling.py

## Bước 5: Phân tích sau Tiling — Kiểm tra Data Explosion

In [ ]:
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict

BASE_DIR  = Path("/kaggle/working/waste-detection_project")
PROCESSED = BASE_DIR / "data" / "processed"
TILED     = BASE_DIR / "data" / "processed_tiled"
SPLITS    = ["train", "val", "test"]
IMG_EXTS  = {".jpg", ".jpeg", ".png"}

stats = {}
for split in SPLITS:
    orig_imgs = [p for p in (PROCESSED / "images" / split).rglob("*")
                 if p.suffix.lower() in IMG_EXTS]
    tile_imgs = [p for p in (TILED / "images" / split).rglob("*")
                 if p.suffix.lower() in IMG_EXTS]

    tiles_with_obj = 0
    tiles_empty    = 0
    obj_counts     = []
    tiles_per_image = defaultdict(int)

    for tp in tile_imgs:
        parts     = tp.stem.rsplit("_tile_", 1)
        orig_stem = parts[0] if len(parts) == 2 else tp.stem
        tiles_per_image[orig_stem] += 1

        lbl_path = (TILED / "labels" / split
                    / tp.parent.relative_to(TILED / "images" / split)
                    / (tp.stem + ".txt"))
        if lbl_path.exists():
            lines = [l.strip() for l in lbl_path.read_text().splitlines() if l.strip()]
            if lines:
                tiles_with_obj += 1
                obj_counts.append(len(lines))
            else:
                tiles_empty += 1
        else:
            tiles_empty += 1

    stats[split] = {
        "orig"           : len(orig_imgs),
        "tiles"          : len(tile_imgs),
        "expansion"      : len(tile_imgs) / max(len(orig_imgs), 1),
        "with_obj"       : tiles_with_obj,
        "empty"          : tiles_empty,
        "obj_counts"     : obj_counts,
        "tiles_per_image": list(tiles_per_image.values()),
    }

print("=" * 70)
print("  KẾT QUẢ TILING (sau anti-overfitting)")
print("=" * 70)
print(f"{'Split':<8} {'Ảnh gốc':>10} {'Tiles':>10} {'Hệ số':>8} {'Có obj':>9} {'Trống':>8} {'%Trống':>8}")
print("-" * 70)
for split in SPLITS:
    s = stats[split]
    pct_empty = s['empty'] / max(s['tiles'], 1) * 100
    flag = "  ✅" if s['expansion'] <= 15 else "  ⚠️"
    print(f"{split:<8} {s['orig']:>10,} {s['tiles']:>10,} {s['expansion']:>7.1f}x {s['with_obj']:>9,} {s['empty']:>8,} {pct_empty:>7.1f}%{flag}")
print("=" * 70)

In [ ]:
# ── Visualisation: so sánh trước/sau anti-overfitting ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Kiểm tra Tiling sau Anti-Overfitting (INCLUDE_EMPTY=False)",
             fontsize=13, fontweight="bold")

# Panel 1: Hệ số mở rộng
ax = axes[0]
expansions = [stats[s]["expansion"] for s in SPLITS]
colors = ["#27ae60" if e <= 15 else "#f39c12" if e <= 30 else "#e74c3c" for e in expansions]
bars = ax.bar(SPLITS, expansions, color=colors, edgecolor="white", linewidth=1.2)
ax.axhline(15, color="orange", linestyle="--", linewidth=1, label="ngưỡng lý tưởng (15x)")
ax.axhline(48, color="red",    linestyle=":",  linewidth=1, label="config cũ (~48x)")
for bar, val in zip(bars, expansions):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{val:.1f}x", ha="center", va="bottom", fontweight="bold", fontsize=11)
ax.set_title("Hệ số mở rộng dữ liệu")
ax.set_ylabel("Tiles / Ảnh gốc")
ax.legend(fontsize=8)
ax.set_ylim(0, max(max(expansions) * 1.3, 55))

# Panel 2: Tỷ lệ tiles có object vs trống (train)
ax = axes[1]
tr = stats["train"]
if tr["empty"] > 0:
    wedge_data   = [tr["with_obj"], tr["empty"]]
    wedge_labels = [f"Có object\n({tr['with_obj']:,})", f"Trống\n({tr['empty']:,})"]
    wedge_colors = ["#2ecc71", "#e74c3c"]
    ax.pie(wedge_data, labels=wedge_labels, colors=wedge_colors,
           autopct="%1.1f%%", startangle=90, textprops={"fontsize": 10})
else:
    ax.pie([1], labels=[f"Tất cả có object\n({tr['with_obj']:,} tiles)"],
           colors=["#2ecc71"], startangle=90, textprops={"fontsize": 11})
ax.set_title(f"[Train] Tiles có object vs trống\n(tổng: {tr['tiles']:,} tiles)")

# Panel 3: Phân phối tiles/ảnh gốc (train)
ax = axes[2]
tpi = stats["train"]["tiles_per_image"]
if tpi:
    ax.hist(tpi, bins=range(min(tpi), max(tpi)+2), color="#3498db",
            edgecolor="white", linewidth=0.7)
    ax.axvline(np.mean(tpi), color="red", linestyle="--", linewidth=1.5,
               label=f"mean = {np.mean(tpi):.1f}")
    ax.set_title("Tiles có object / ảnh gốc (Train)")
    ax.set_xlabel("Số tiles (chỉ tiles có object)")
    ax.set_ylabel("Số ảnh")
    ax.legend(fontsize=9)

plt.tight_layout()
RESULTS = BASE_DIR / "results"
RESULTS.mkdir(exist_ok=True)
plt.savefig(RESULTS / "tiling_anti_overfitting_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("[Saved] tiling_anti_overfitting_analysis.png")

In [ ]:
# ── Đánh giá nguy cơ overfitting (sau khi áp dụng fix) ───────────────────────
print("\n" + "=" * 65)
print("  ĐÁNH GIÁ NGUY CƠ OVERFITTING (sau anti-overfitting)")
print("=" * 65)

tr = stats["train"]
expansion_train = tr["expansion"]
pct_empty_train = tr["empty"] / max(tr["tiles"], 1) * 100
mean_tpi = np.mean(tr["tiles_per_image"]) if tr["tiles_per_image"] else 0

def risk_level(expansion, pct_empty, mean_tpi):
    risks = []
    # Risk 1: Data explosion
    if expansion > 20:
        risks.append(("Data Explosion", "🔴 CAO",
                      f"Hệ số {expansion:.1f}x vẫn còn cao. Cân nhắc giảm epoch."))
    elif expansion > 10:
        risks.append(("Data Explosion", "🟡 TRUNG BÌNH",
                      f"Hệ số {expansion:.1f}x chấp nhận được. Theo dõi val loss."))
    else:
        risks.append(("Data Explosion", "🟢 THẤP",
                      f"Hệ số {expansion:.1f}x — mức an toàn."))

    # Risk 2: Background dominance
    if pct_empty > 30:
        risks.append(("Background Tiles", "🟠 TRUNG BÌNH",
                      f"{pct_empty:.1f}% tiles trống. Cân nhắc giảm thêm."))
    elif pct_empty > 0:
        risks.append(("Background Tiles", "🟢 THẤP",
                      f"Chỉ {pct_empty:.1f}% tiles trống — trong ngưỡng an toàn."))
    else:
        risks.append(("Background Tiles", "🟢 RẤT THẤP",
                      "Không có tiles trống (INCLUDE_EMPTY=False)."))

    # Risk 3: Tile correlation
    if mean_tpi > 15:
        risks.append(("Tile Correlation", "🟠 TRUNG BÌNH",
                      f"Mean {mean_tpi:.1f} tiles/ảnh — tiles vẫn có tương quan nhất định."))
    else:
        risks.append(("Tile Correlation", "🟢 THẤP",
                      f"Mean {mean_tpi:.1f} tiles/ảnh — mức hợp lý."))

    return risks

for name, level, desc in risk_level(expansion_train, pct_empty_train, mean_tpi):
    print(f"\n  [{name}]")
    print(f"    Mức độ : {level}")
    print(f"    Chi tiết: {desc}")

print("\n" + "=" * 65)

## Bước 6: Huấn luyện YOLOv8n trên dữ liệu Tiled

> Giữ nguyên 100% hyperparameter của baseline. Điểm khác biệt DUY NHẤT là dataset.

In [ ]:
import os, time, glob
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

BASE_DIR        = Path("/kaggle/working/waste-detection_project")
YAML_PATH_TILED = BASE_DIR / "data" / "processed_tiled" / "dataset.yaml"
RESULTS         = BASE_DIR / "results"
RESULTS.mkdir(exist_ok=True)

print("=" * 60)
print("  EXP 6: YOLOv8n + Tiling (Anti-Overfitting)")
print("=" * 60)
print(f"  Dataset : {YAML_PATH_TILED}")
print(f"  Model   : YOLOv8n")
print(f"  Epochs  : 50  ← giữ nguyên baseline")
print(f"  imgsz   : 640 ← giữ nguyên baseline")
print(f"  batch   : 16  ← giữ nguyên baseline")
print("=" * 60)

model_path = BASE_DIR / "models" / "yolov8n.pt"
model = YOLO(str(model_path))

train_results = model.train(
    data     = str(YAML_PATH_TILED),
    epochs   = 50,
    imgsz    = 640,
    batch    = 16,
    name     = "exp6_tiling_yolov8n",
    project  = str(RESULTS / "runs"),
    exist_ok = True,
    verbose  = True,
)

## Bước 7: Đánh giá & So sánh với Baseline

In [ ]:
print("\n[Evaluation] Đánh giá trên tập Test (tiled)...")
metrics = model.val(split="test", verbose=False)

map50   = float(metrics.box.map50)
prec    = float(metrics.box.mp)
recall  = float(metrics.box.mr)
map5095 = float(metrics.box.map)

print(f"  mAP@0.5      : {map50:.4f}  ({map50*100:.2f}%)")
print(f"  mAP@0.5:0.95 : {map5095:.4f}  ({map5095*100:.2f}%)")
print(f"  Precision    : {prec:.4f}  ({prec*100:.2f}%)")
print(f"  Recall       : {recall:.4f}  ({recall*100:.2f}%)")

val_imgs = glob.glob(
    str(BASE_DIR / "data/processed_tiled/images/test/**/*.*"), recursive=True
)[:50]
if val_imgs:
    t0 = time.perf_counter()
    for img_path in val_imgs:
        model.predict(img_path, verbose=False)
    elapsed = (time.perf_counter() - t0) / len(val_imgs) * 1000
    print(f"  Inference (avg): {elapsed:.1f} ms/tile")
else:
    elapsed = None

result_row = {
    "Model"            : "YOLOv8n + Tiling (50ep, INCLUDE_EMPTY=False)",
    "mAP@0.5 (%)"      : round(map50    * 100, 2),
    "mAP@0.5:0.95 (%)" : round(map5095  * 100, 2),
    "Precision (%)"    : round(prec     * 100, 2),
    "Recall (%)"       : round(recall   * 100, 2),
    "Inference (ms)"   : round(elapsed, 1) if elapsed else "N/A",
    "Train Tiles"      : stats["train"]["tiles"],
    "Expansion"        : f"{stats['train']['expansion']:.1f}x",
    "Empty Tile %"     : f"{stats['train']['empty'] / max(stats['train']['tiles'],1)*100:.1f}%",
    "INCLUDE_EMPTY"    : False,
    "MAX_EMPTY_RATIO"  : 0.5,
}
pd.DataFrame([result_row]).to_csv(RESULTS / "ket_qua_exp6_tiling.csv", index=False)
print("\n[Saved] ket_qua_exp6_tiling.csv")

In [ ]:
# ── So sánh với Baseline ──────────────────────────────────────────────────────
baseline_csv = RESULTS / "ket_qua_baseline.csv"

print("\n" + "=" * 65)
print("  SO SÁNH: BASELINE vs EXP6 (Tiling + Anti-Overfitting)")
print("=" * 65)

exp6_row = {
    "Experiment"     : "Exp6: YOLOv8n + Tiling (AO)",
    "mAP@0.5 (%)"    : round(map50  * 100, 2),
    "Precision (%)"  : round(prec   * 100, 2),
    "Recall (%)"     : round(recall * 100, 2),
    "Inference (ms)" : round(elapsed, 1) if elapsed else "N/A",
    "Ghi chú"        : f"Tiles={stats['train']['tiles']:,} | {stats['train']['expansion']:.1f}x | Empty=0%",
}

rows = []
if baseline_csv.exists():
    df_base = pd.read_csv(baseline_csv)
    baseline_row = {
        "Experiment"     : "Baseline: YOLOv8n",
        "mAP@0.5 (%)"    : float(df_base["mAP@0.5 (%)"].iloc[0]),
        "Precision (%)"  : float(df_base["Precision (%)"].iloc[0]),
        "Recall (%)"     : float(df_base["Recall (%)"].iloc[0]),
        "Inference (ms)" : df_base["Inference (ms)"].iloc[0],
        "Ghi chú"        : "Ảnh gốc, không tiling",
    }
    rows = [baseline_row, exp6_row]
    delta_map = exp6_row["mAP@0.5 (%)"] - baseline_row["mAP@0.5 (%)"]
    df_cmp = pd.DataFrame(rows)
    print(df_cmp.to_string(index=False))
    print(f"\n  Δ mAP@0.5 : {delta_map:+.2f}%", end="")
    if delta_map > 1:
        print(" ✅  Tiling CẢI THIỆN hiệu năng")
    elif delta_map < -1:
        print(" ❌  Tiling LÀM GIẢM hiệu năng (kiểm tra overfitting)")
    else:
        print(" ≈   Tiling không có tác động đáng kể")
else:
    rows = [exp6_row]
    print("[INFO] Không tìm thấy baseline CSV. Kết quả Exp6:")
    print(pd.DataFrame(rows).to_string(index=False))

pd.DataFrame(rows).to_csv(RESULTS / "so_sanh_baseline_vs_exp6_tiling.csv", index=False)
print("\n[Saved] so_sanh_baseline_vs_exp6_tiling.csv")

In [ ]:
# ── Training Curves — kiểm tra overfitting ────────────────────────────────────
import pandas as pd

results_csv = RESULTS / "runs" / "exp6_tiling_yolov8n" / "results.csv"

if results_csv.exists():
    df_train = pd.read_csv(results_csv)
    df_train.columns = df_train.columns.str.strip()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Exp6: Training Curves — Kiểm tra Overfitting",
                 fontsize=13, fontweight="bold")

    # Panel 1: Box Loss
    ax = axes[0]
    if "train/box_loss" in df_train.columns:
        ax.plot(df_train["epoch"], df_train["train/box_loss"],
                label="Train Box Loss", color="#e74c3c", linewidth=1.5)
    if "val/box_loss" in df_train.columns:
        ax.plot(df_train["epoch"], df_train["val/box_loss"],
                label="Val Box Loss", color="#3498db", linestyle="--", linewidth=1.5)

    # Phát hiện overfitting: val loss tăng khi train loss giảm (10 epoch cuối)
    if "val/box_loss" in df_train.columns and len(df_train) > 10:
        tail = df_train.tail(10)
        if (tail["val/box_loss"].iloc[-1] > tail["val/box_loss"].iloc[0] and
                tail["train/box_loss"].iloc[-1] < tail["train/box_loss"].iloc[0]):
            overfit_epoch = df_train["epoch"].iloc[-10]
            ax.axvline(overfit_epoch, color="purple", linestyle=":",
                       linewidth=2, label=f"⚠ Overfit ~ep{overfit_epoch:.0f}")
            print(f"⚠️  [CẢNH BÁO] Val loss tăng trong 10 epoch cuối → dấu hiệu OVERFITTING!")
        else:
            print("✅  Không phát hiện dấu hiệu overfitting rõ ràng trong training curves.")

    ax.set_title("Box Loss (Train vs Val)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # Panel 2: Cls Loss
    ax = axes[1]
    for col, label, color, ls in [
        ("train/cls_loss", "Train Cls Loss", "#e74c3c", "-"),
        ("val/cls_loss",   "Val Cls Loss",   "#3498db", "--"),
    ]:
        if col in df_train.columns:
            ax.plot(df_train["epoch"], df_train[col], label=label,
                    color=color, linestyle=ls, linewidth=1.5)
    ax.set_title("Class Loss (Train vs Val)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # Panel 3: mAP@0.5
    ax = axes[2]
    map_col = next((c for c in ["metrics/mAP50(B)", "metrics/mAP_0.5", "metrics/mAP50"]
                    if c in df_train.columns), None)
    if map_col:
        ax.plot(df_train["epoch"], df_train[map_col] * 100,
                label="mAP@0.5", color="#27ae60", linewidth=1.5)
        # Đánh dấu best epoch
        best_ep = df_train[map_col].idxmax()
        ax.axvline(df_train.loc[best_ep, "epoch"], color="orange", linestyle="--",
                   linewidth=1.2,
                   label=f"Best ep{df_train.loc[best_ep,'epoch']:.0f} ({df_train.loc[best_ep,map_col]*100:.1f}%)")
    ax.set_title("mAP@0.5 theo Epoch")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("mAP@0.5 (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(RESULTS / "exp6_training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("[Saved] exp6_training_curves.png")
else:
    print(f"[WARN] Không tìm thấy results.csv tại: {results_csv}")

In [ ]:
# ── Tổng kết ──────────────────────────────────────────────────────────────────
print("=" * 65)
print("  TỔNG KẾT EXP6: TILING + ANTI-OVERFITTING")
print("=" * 65)

tr = stats["train"]
print(f"""
▸ Tiling config:
    INCLUDE_EMPTY = False  (loại bỏ toàn bộ tiles trống)
    OVERLAP       = 20%    (stride = 512px)
    TILE_SIZE     = 640    (khớp imgsz)

▸ Dữ liệu sau tiling:
    Ảnh gốc train : {tr['orig']:,}
    Tiles train   : {tr['tiles']:,}  ({tr['expansion']:.1f}x)
    Tiles có obj  : {tr['with_obj']:,}  (100% tiles train)
    Tiles trống   : {tr['empty']:,}   (đã loại bỏ)

▸ Kết quả model:
    mAP@0.5      : {map50*100:.2f}%
    mAP@0.5:0.95 : {map5095*100:.2f}%
    Precision    : {prec*100:.2f}%
    Recall       : {recall*100:.2f}%

▸ Nguy cơ overfitting:
    Hệ số expansion {tr['expansion']:.1f}x (giảm từ ~48x xuống nhờ loại empty tiles)
    → Xem training curves bên trên để kết luận chính xác.
""")